# Member A -- Agreement Network Between Students

Nodes = students, edges = similarity of per-item z-scored opinion vectors.

**Data source note (explicit user instruction, deviates from CLAUDE.md):**
This analysis reads `data/clean/cleaned_Survey_Results_UC.csv` (85 respondents,
rows with any blank cell already dropped) instead of reading
`data/raw/Survey_Results_UC.csv` fresh and applying the project's general
>30-missing-per-row rule (which would give 87 respondents). This was a
deliberate choice by the user for this member's folder only; member_b uses
the 87-respondent path. See `FINDINGS.md` for the numbers this produces and
why it matters for the final cross-member comparison in the report.

In [1]:
import sys
sys.path.insert(0, 'src')

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from load import load_clean, zscore_items, zscore_items_mean_imputed, SEED, N_RESPONDENTS_EXPECTED
from build import (
    pairwise_complete_cosine, mean_imputed_cosine,
    build_threshold_graph, build_mutual_knn_graph, build_disparity_backbone,
    percolation_sweep, graph_summary, edge_overlap,
    SIM_THRESHOLD, KNN_K, DISPARITY_ALPHA,
)
from metrics import (
    clustering_metrics, centralities, detect_communities, modularity,
    null_model_comparison, log_binned_degree_histogram,
)
from dynamics import bounded_confidence_sweep, find_critical_epsilon, si_seeding_comparison

FIG_DIR = 'figures'
TAB_DIR = 'tables'

## 1. Load and z-score

In [2]:
coded = load_clean()
print(f'Respondents x items: {coded.shape} (expected {N_RESPONDENTS_EXPECTED} x 60)')
z = zscore_items(coded)

raw_corr = np.corrcoef(coded.fillna(coded.mean()).to_numpy())
iu = np.triu_indices(len(coded), 1)
print(f'Raw (non-zscored) pairwise correlation: mean={raw_corr[iu].mean():.3f}, min={raw_corr[iu].min():.3f}')

sim, shared = pairwise_complete_cosine(z)
svals = sim.to_numpy()[iu]
print(f'z-scored pairwise-complete cosine similarity: mean={svals.mean():.3f}, min={svals.min():.3f}, max={svals.max():.3f}')

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(svals, bins=30, color='#4C72B0', edgecolor='white')
ax.set_xlabel('Pairwise cosine similarity (z-scored items)')
ax.set_ylabel('Count')
ax.set_title('Similarity distribution after per-item z-scoring')
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/fig01_similarity_distribution.png', dpi=150)
plt.close(fig)

Respondents x items: (85, 60) (expected 85 x 60)
Raw (non-zscored) pairwise correlation: mean=0.274, min=-0.359
z-scored pairwise-complete cosine similarity: mean=-0.010, min=-0.634, max=0.752


## 2. Edge-rule comparison: threshold vs. mutual-kNN vs. disparity backbone

In [3]:
G_thresh = build_threshold_graph(sim, SIM_THRESHOLD)
G_knn = build_mutual_knn_graph(sim, KNN_K)
G_backbone = build_disparity_backbone(sim, DISPARITY_ALPHA, SIM_THRESHOLD)

summary_rows = []
for name, G in [('threshold', G_thresh), ('mutual_knn', G_knn), ('disparity_backbone', G_backbone)]:
    s = graph_summary(G)
    s['rule'] = name
    summary_rows.append(s)
edge_rule_df = pd.DataFrame(summary_rows)[['rule', 'n_nodes', 'n_edges', 'mean_degree', 'density', 'giant_component_fraction', 'n_isolates']]
edge_rule_df.to_csv(f'{TAB_DIR}/table01_edge_rule_comparison.csv', index=False)
print(edge_rule_df.to_string(index=False))

print(f"\nEdge overlap threshold vs kNN (Jaccard): {edge_overlap(G_thresh, G_knn):.3f}")
print(f"Edge overlap threshold vs backbone (Jaccard): {edge_overlap(G_thresh, G_backbone):.3f}")

              rule  n_nodes  n_edges  mean_degree  density  giant_component_fraction  n_isolates
         threshold       85      775    18.235294 0.217087                  1.000000           0
        mutual_knn       85      203     4.776471 0.056863                  1.000000           0
disparity_backbone       85        2     0.047059 0.000560                  0.023529          81

Edge overlap threshold vs kNN (Jaccard): 0.262
Edge overlap threshold vs backbone (Jaccard): 0.003


## 3. Percolation sweep -- giant component emergence

In [4]:
sweep = percolation_sweep(sim)
sweep.to_csv(f'{TAB_DIR}/table02_percolation_sweep.csv', index=False)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(sweep['threshold'], sweep['giant_component_fraction'], marker='o', ms=3, color='#DD8452')
ax.axvline(SIM_THRESHOLD, color='gray', linestyle='--', label=f'primary threshold={SIM_THRESHOLD}')
ax.set_xlabel('Similarity threshold')
ax.set_ylabel('Giant component fraction')
ax.set_title('Percolation: giant component vs. similarity threshold')
ax.legend()
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/fig02_percolation_sweep.png', dpi=150)
plt.close(fig)

half = sweep[sweep['giant_component_fraction'] >= 0.5]
print(f"Giant component reaches >=50% of nodes at threshold >= {half.iloc[-1]['threshold'] if len(half) else 'never'}")

Giant component reaches >=50% of nodes at threshold >= 0.325


## 4. Degree distribution (log-binned), clustering, centralities

In [5]:
G = G_thresh  # primary graph for the rest of the analysis

hist = log_binned_degree_histogram(G)
hist.to_csv(f'{TAB_DIR}/table03_degree_histogram_logbinned.csv', index=False)

fig, ax = plt.subplots(figsize=(6, 4))
pos = hist[hist['bin_low'] > 0]
ax.loglog(pos['bin_low'], pos['density'], marker='o', ms=4, color='#55A868')
ax.set_xlabel('Degree (log)')
ax.set_ylabel('Density (log)')
ax.set_title('Log-binned degree distribution (threshold graph)')
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/fig03_degree_distribution_logbinned.png', dpi=150)
plt.close(fig)

cm = clustering_metrics(G)
print('Clustering metrics:', cm)

cent = centralities(G)
cent.to_csv(f'{TAB_DIR}/table04_centralities.csv', index=False)
print('\nTop 5 eigenvector centrality:')
print(cent.head(5).to_string(index=False))
print('\nTop 5 betweenness centrality:')
print(cent.sort_values('betweenness', ascending=False).head(5).to_string(index=False))

Clustering metrics: {'global_clustering_transitivity': 0.5741161206616932, 'average_local_clustering': 0.4946151572877853, 'n_connected_components': 1, 'avg_shortest_path_giant_component': 2.062745098039216}

Top 5 eigenvector centrality:
node  degree  closeness  betweenness  eigenvector
 110      32   0.528302     0.011997     0.223098
 114      31   0.506024     0.007502     0.220379
  94      31   0.518519     0.010000     0.217305
  81      31   0.531646     0.021944     0.212401
  54      29   0.502994     0.006832     0.209503

Top 5 betweenness centrality:
node  degree  closeness  betweenness  eigenvector
  72      14   0.521739     0.040405     0.060818
  57      17   0.515337     0.039051     0.094949
  55      28   0.521739     0.030165     0.189248
  19      18   0.509091     0.029723     0.092988
  82      21   0.491228     0.025604     0.146027


## 5. Community detection and Erdos-Renyi null model comparison

In [6]:
comm_map, comms = detect_communities(G)
mod = modularity(G, comms)
print(f'Louvain communities: {len(comms)}, sizes: {sorted([len(c) for c in comms], reverse=True)}')
print(f'Modularity: {mod:.4f}')

comm_df = pd.DataFrame({'node': list(comm_map.keys()), 'community': list(comm_map.values())})
comm_df.to_csv(f'{TAB_DIR}/table05_communities.csv', index=False)

null_cmp = null_model_comparison(G)
null_cmp.to_csv(f'{TAB_DIR}/table06_null_model_comparison.csv', index=False)
print('\nObserved vs ER null (n, mean-degree matched), mean +/- SD over 30 seeded realizations:')
print(null_cmp.to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 4))
pos_layout = nx.spring_layout(G, seed=SEED)
colors = [comm_map[n] for n in G.nodes()]
nx.draw_networkx_nodes(G, pos_layout, node_color=colors, cmap='tab10', node_size=60, ax=ax)
nx.draw_networkx_edges(G, pos_layout, alpha=0.15, ax=ax)
ax.set_title(f'Agreement network, colored by Louvain community (Q={mod:.3f})')
ax.axis('off')
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/fig04_network_communities.png', dpi=150)
plt.close(fig)

Louvain communities: 4, sizes: [46, 35, 2, 2]
Modularity: 0.4277



Observed vs ER null (n, mean-degree matched), mean +/- SD over 30 seeded realizations:
                           metric  observed  null_mean  null_sd
         average_local_clustering  0.494615   0.217701 0.009051
avg_shortest_path_giant_component  2.062745   1.797507 0.012344
         giant_component_fraction  1.000000   1.000000 0.000000


## 6. Bounded-confidence opinion dynamics (Deffuant-Weisbuch)

In [7]:
bc_sweep = bounded_confidence_sweep(G, n_trials=30)
bc_sweep.to_csv(f'{TAB_DIR}/table07_bounded_confidence_sweep.csv', index=False)
crit_eps = find_critical_epsilon(bc_sweep)
print(f'Critical epsilon (fragmentation -> consensus, >=50% trials): {crit_eps}')

fig, ax = plt.subplots(figsize=(6, 4))
ax.errorbar(bc_sweep['epsilon'], bc_sweep['mean_n_clusters'], yerr=bc_sweep['sd_n_clusters'],
            marker='o', ms=3, color='#C44E52', ecolor='lightgray', capsize=2)
if crit_eps is not None:
    ax.axvline(crit_eps, color='gray', linestyle='--', label=f'critical epsilon={crit_eps}')
ax.set_xlabel('Confidence threshold (epsilon)')
ax.set_ylabel('Final number of opinion clusters (mean +/- SD, 30 trials)')
ax.set_title('Deffuant-Weisbuch: fragmentation -> consensus transition')
ax.legend()
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/fig05_bounded_confidence_transition.png', dpi=150)
plt.close(fig)

Critical epsilon (fragmentation -> consensus, >=50% trials): 0.4


## 7. SI spreading: high-eigenvector-centrality vs. random seeding

In [8]:
hc_nodes = cent.sort_values('eigenvector', ascending=False)['node'].tolist()
si_cmp = si_seeding_comparison(G, hc_nodes, n_trials=30)
si_cmp.to_csv(f'{TAB_DIR}/table08_si_seeding_comparison.csv', index=False)
print(si_cmp.to_string(index=False))

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(si_cmp['seeding'], si_cmp['mean_time_to_full_coverage'], yerr=si_cmp['sd_time_to_full_coverage'],
       color=['#4C72B0', '#DD8452'], capsize=4)
ax.set_ylabel('Mean time to full coverage (steps, 30 trials)')
ax.set_title('SI spreading: seeding strategy comparison')
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/fig06_si_seeding_comparison.png', dpi=150)
plt.close(fig)

                    seeding  mean_final_coverage  sd_final_coverage  frac_trials_full_coverage  mean_time_to_full_coverage  sd_time_to_full_coverage
high_eigenvector_centrality             0.996863           0.006024                   0.766667                    9.000000                  0.932505
                     random             0.996863           0.005203                   0.733333                    8.590909                  1.230635
